In [1]:
import numpy as np
import pickle

In [2]:
dense_embeddings = np.load("../data/processed/_dense_sparse_court/0.npy")
print(dense_embeddings.shape)

(10000, 1024)


In [3]:
with open("../data/processed/_dense_sparse_court/0.pkl", 'rb') as inf:
    sparse_l = pickle.load(inf)

print(len(sparse_l))
print(sparse_l[0])

10000
{'Ver': np.float16(0.0788), 'we': np.float16(0.141), 'iger': np.float16(0.1787), 'Bei': np.float16(0.1316), 'la': np.float16(0.1881), 'dung': np.float16(0.06366), 'seien': np.float16(0.0958), 'gut': np.float16(0.1322), 'zu': np.float16(0.074), 'hei': np.float16(0.1565), 'ssen': np.float16(0.0874), '.': np.float16(0.011566), 'GE': np.float16(0.1143), '139': np.float16(0.1785), 'I': np.float16(0.0438), '2': np.float16(0.0772), '7': np.float16(0.1049)}


In [1]:
import json

count = 0
with open('../ft_data/train.jsonl', 'r', encoding='utf-8') as inf:
    for line in inf:
        j = json.loads(line.strip())
        if len(j['neg']) != 5:
            count += 1
print(count)

246


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

path = "/root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/ft_data/bge-reranker-v2-m3-finetune"

tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path)
model.eval()

# 验证推理
inputs = tokenizer("这是一个查询", "这是一个段落", return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    logits = model(**inputs).logits
print("logits:", logits)  # 能输出数值就说明权重正常

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

inputs_args = ("法律合同纠纷", "本合同受中华人民共和国法律管辖")

# 微调后
path_ft = "/root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/ft_data/bge-reranker-v2-m3-finetune"
tokenizer = AutoTokenizer.from_pretrained(path_ft)
model_ft = AutoModelForSequenceClassification.from_pretrained(path_ft)
inputs = tokenizer(*inputs_args, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    print("微调后 logits:", model_ft(**inputs).logits)

# 原始模型
path_base = "/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3"
model_base = AutoModelForSequenceClassification.from_pretrained(path_base)
with torch.no_grad():
    print("原始模型 logits:", model_base(**inputs).logits)

In [7]:
from safetensors.torch import load_file

state_dict = load_file("../ft_data/bge-reranker-v2-m3-finetune/model.safetensors")
print(len(state_dict))
print(state_dict)

FileNotFoundError: No such file or directory: ../ft_data/bge-reranker-v2-m3-finetune/model.safetensors

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained("../ft_data/lora_output")
base = AutoModelForSequenceClassification.from_pretrained(
    "/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3'", num_labels=1
)
model = PeftModel.from_pretrained(base, "../ft_data/lora_reranker_output")
model.eval()

query   = "什么是向量数据库？"
passage = "向量数据库专门用于存储和检索高维向量..."

inputs = tokenizer(query, passage, return_tensors="pt", truncation=True, max_length=512)
with torch.no_grad():
    score = model(**inputs).logits.item()

print(f"相关性分数: {score:.4f}")